# 🧬 02. 전처리 (Preprocessing)
> **fertility-predictor** | 난임 환자 임신 성공 여부 예측  
> 데이터: train.csv (256,351행) / test.csv (90,067행) / 68개 피처

| 단계 | 내용 |
|------|------|
| 1 | 시술 횟수 컬럼 수치 변환 (`6회 이상` → 6) |
| 2 | 특정 시술 유형 멀티핫 인코딩 |
| 3 | DI 구조적 결측치 0 채움 |
| 4 | 파생 피처 생성 (수정률·이식률·배양기간·시술_임신_비율·has_blastocyst) |
| 5 | 수치형 결측치: 시술 유형별 그룹 중앙값 (train fit → test transform) |
| 6 | 범주형 인코딩 (Ordinal / Label) |
| 7 | 검증 및 pkl 저장 |

## 0. 라이브러리 및 설정

In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

TARGET     = '임신 성공 여부'
DATA_DIR   = '../data'
OUTPUT_DIR = '../data'

print('✅ 라이브러리 로드 완료')

✅ 라이브러리 로드 완료


## 1. 데이터 로드

In [2]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f'Train shape : {train.shape}')
print(f'Test  shape : {test.shape}')
train.head(3)

Train shape : (256351, 69)
Test  shape : (90067, 68)


,ID,시술 시기 코드,시술 당시 나이,임신 시도 또는 마지막 임신 경과 연수,시술 유형,특정 시술 유형,배란 자극 여부,배란 유도 유형,단일 배아 이식 여부,착상 전 유전 검사 사용 여부,...,기증 배아 사용 여부,대리모 여부,PGD 시술 여부,PGS 시술 여부,난자 채취 경과일,난자 해동 경과일,난자 혼합 경과일,배아 이식 경과일,배아 해동 경과일,임신 성공 여부
0,TRAIN_000000,TRZKPL,만18-34세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,3.0,NaN,0
1,TRAIN_000001,TRYBLT,만45-50세,NaN,IVF,ICSI,0,알 수 없음,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,0
2,TRAIN_000002,TRVNRY,만18-34세,NaN,IVF,IVF,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,2.0,NaN,0


## 2. 시술 횟수 컬럼 수치 변환
> `'0회'` ~ `'6회 이상'` → 정수 (6회 이상 = 6)

In [3]:
COUNT_COLS = [
    '총 시술 횟수', '클리닉 내 총 시술 횟수',
    'IVF 시술 횟수', 'DI 시술 횟수',
    '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
    '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수',
]
COUNT_COLS = [c for c in COUNT_COLS if c in train.columns]

def parse_count(val):
    """'N회' 또는 'N회 이상' → int, NaN 유지"""
    if pd.isna(val):
        return np.nan
    s = str(val).replace('회 이상', '').replace('회', '').strip()
    try:
        return int(s)
    except ValueError:
        return np.nan

for col in COUNT_COLS:
    train[col] = train[col].apply(parse_count)
    test[col]  = test[col].apply(parse_count)

print('✅ 시술 횟수 컬럼 수치 변환 완료')
print(f'변환 컬럼: {COUNT_COLS}')
train[COUNT_COLS].head(3)

✅ 시술 횟수 컬럼 수치 변환 완료
변환 컬럼: ['총 시술 횟수', '클리닉 내 총 시술 횟수', 'IVF 시술 횟수', 'DI 시술 횟수', '총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수', '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수']


,총 시술 횟수,클리닉 내 총 시술 횟수,IVF 시술 횟수,DI 시술 횟수,총 임신 횟수,IVF 임신 횟수,DI 임신 횟수,총 출산 횟수,IVF 출산 횟수,DI 출산 횟수
0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0
2,1,0,1,0,0,0,0,0,0,0


## 3. 특정 시술 유형 — 멀티핫 인코딩
> 복합 문자열(`'ICSI / BLASTOCYST:IVF / BLASTOCYST'` 등)에서 키워드 추출

In [4]:
PROC_KEYWORDS = ['IVF', 'ICSI', 'IUI', 'FER', 'BLASTOCYST', 'AH', 'GIFT', 'ICI', 'IVI']

def add_procedure_multihot(df, keywords=PROC_KEYWORDS):
    src = '특정 시술 유형'
    for kw in keywords:
        df[f'proc_{kw}'] = df[src].str.contains(kw, na=False).astype(np.int8)
    return df

train = add_procedure_multihot(train)
test  = add_procedure_multihot(test)

mhot_cols = [f'proc_{kw}' for kw in PROC_KEYWORDS]
print('✅ 특정 시술 유형 멀티핫 인코딩 완료')
pd.DataFrame({
    'train_count': train[mhot_cols].sum(),
    'test_count':  test[mhot_cols].sum(),
})

✅ 특정 시술 유형 멀티핫 인코딩 완료


,train_count,test_count
proc_IVF,95845,33725
proc_ICSI,128547,45383
proc_IUI,6100,2114
proc_FER,3,0
proc_BLASTOCYST,2868,975
proc_AH,1092,369
proc_GIFT,2,0
proc_ICI,96,30
proc_IVI,23,8


## 4. DI 구조적 결측치 처리
> DI(인공수정) 환자는 배아·난자 관련 시술을 받지 않으므로 해당 컬럼의 NaN은 정상 결측 → **0** 으로 채움

In [5]:
IVF_ONLY_COLS = [
    '총 생성 배아 수', '미세주입된 난자 수', '미세주입에서 생성된 배아 수',
    '이식된 배아 수', '미세주입 배아 이식 수', '저장된 배아 수',
    '미세주입 후 저장된 배아 수', '해동된 배아 수', '해동 난자 수',
    '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수',
    '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수',
    '동결 배아 사용 여부', '신선 배아 사용 여부', '기증 배아 사용 여부',
    'PGD 시술 여부', 'PGS 시술 여부',
    '착상 전 유전 검사 사용 여부', '착상 전 유전 진단 사용 여부',
    '단일 배아 이식 여부',
    '난자 채취 경과일', '난자 혼합 경과일', '배아 이식 경과일',
    '난자 해동 경과일', '배아 해동 경과일',
]
IVF_ONLY_COLS = [c for c in IVF_ONLY_COLS if c in train.columns]

for df in [train, test]:
    di_mask = df['시술 유형'] == 'DI'
    df.loc[di_mask, IVF_ONLY_COLS] = df.loc[di_mask, IVF_ONLY_COLS].fillna(0)

print(f'✅ DI 구조적 결측치 0 채움 완료  (대상 컬럼: {len(IVF_ONLY_COLS)}개)')
print(f'  Train DI 행: {(train["시술 유형"]=="DI").sum():,}')
print(f'  Test  DI 행: {(test["시술 유형"]=="DI").sum():,}')

✅ DI 구조적 결측치 0 채움 완료  (대상 컬럼: 27개)
  Train DI 행: 6,291
  Test  DI 행: 2,176


## 5. 파생 피처 생성

| 피처 | 계산식 | 의미 |
|------|--------|------|
| `수정률` | 총 생성 배아 수 / 혼합된 난자 수 | 난자 대비 배아 생성 효율 |
| `이식률` | 이식된 배아 수 / 총 생성 배아 수 | 배아 대비 이식 비율 |
| `배양기간` | 배아 이식 경과일 − 난자 혼합 경과일 | 배아 배양 일수 (5일↑ = 배반포) |
| `시술_임신_비율` | 총 임신 횟수 / 총 시술 횟수 | 누적 시술 대비 임신 성공 이력 |
| `has_blastocyst` | `proc_BLASTOCYST` | 배반포 사용 여부 |

In [6]:
def add_derived_features(df):
    # 수정률
    df['수정률'] = np.where(
        df['혼합된 난자 수'] > 0,
        df['총 생성 배아 수'] / df['혼합된 난자 수'],
        np.nan,
    )

    # 이식률
    df['이식률'] = np.where(
        df['총 생성 배아 수'] > 0,
        df['이식된 배아 수'] / df['총 생성 배아 수'],
        np.nan,
    )

    # 배양기간 (음수·이상치는 NaN 처리)
    df['배양기간'] = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df.loc[df['배양기간'] < 0, '배양기간'] = np.nan

    # 시술_임신_비율 (총 시술 = 0이면 NaN)
    df['시술_임신_비율'] = np.where(
        df['총 시술 횟수'] > 0,
        df['총 임신 횟수'] / df['총 시술 횟수'],
        np.nan,
    )

    # has_blastocyst: 멀티핫 인코딩 결과 활용
    df['has_blastocyst'] = df['proc_BLASTOCYST'].astype(np.int8)

    return df

train = add_derived_features(train)
test  = add_derived_features(test)

DERIVED_COLS = ['수정률', '이식률', '배양기간', '시술_임신_비율', 'has_blastocyst']
print('✅ 파생 피처 생성 완료')
train[DERIVED_COLS].describe().T[['count', 'mean', 'std', 'min', 'max']].round(3)

✅ 파생 피처 생성 완료


,count,mean,std,min,max
수정률,203410.0,0.662,0.250,0.0,1.25
이식률,196711.0,0.367,0.319,0.0,3.00
배양기간,182085.0,3.665,1.413,0.0,7.00
시술_임신_비율,158752.0,0.161,0.289,0.0,1.00
has_blastocyst,256351.0,0.011,0.105,0.0,1.00


## 6. 수치형 결측치 — 시술 유형 그룹별 중앙값 대체
> **Data Leakage 방지**: train 그룹 중앙값만 계산 → train·test 모두 동일 값으로 채움

In [7]:
num_cols = train.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [c for c in num_cols if c != TARGET]

# train에서 그룹별 중앙값 계산 (fit)
group_medians = {}
for stype in ['IVF', 'DI']:
    mask = train['시술 유형'] == stype
    group_medians[stype] = train.loc[mask, num_cols].median()

print('그룹별 중앙값 계산 완료 (train fit)')

# train & test 각각 그룹 중앙값으로 채움 (transform)
for df in [train, test]:
    for stype in ['IVF', 'DI']:
        mask = df['시술 유형'] == stype
        df.loc[mask, num_cols] = df.loc[mask, num_cols].fillna(group_medians[stype])

# 시술 유형 누락 행 등 잔여 결측치: 전체 train 중앙값으로 처리
global_medians = train[num_cols].median()
train[num_cols] = train[num_cols].fillna(global_medians)
test[num_cols]  = test[num_cols].fillna(global_medians)

print('✅ 수치형 결측치 처리 완료')
print(f'  Train 잔여 결측: {train[num_cols].isnull().sum().sum()}')
print(f'  Test  잔여 결측: {test[num_cols].isnull().sum().sum()}')

그룹별 중앙값 계산 완료 (train fit)
✅ 수치형 결측치 처리 완료
  Train 잔여 결측: 0
  Test  잔여 결측: 0


## 7. 범주형 인코딩
### 7-1. 나이 컬럼 — Ordinal 인코딩

In [8]:
AGE_ORDER = [
    '만18-34세', '만35-37세', '만38-39세',
    '만40-42세', '만43-44세', '만45-50세', '만50세 이상',
]
AGE_MAP = {v: i for i, v in enumerate(AGE_ORDER)}

# 난자·정자 기증자 나이 (공통 구간)
DONOR_AGE_ORDER = [
    '알 수 없음',
    '만18-20세', '만21-25세', '만26-30세', '만31-35세',
    '만36-40세', '만41-45세', '만45세 이상',
]
DONOR_AGE_MAP = {v: i for i, v in enumerate(DONOR_AGE_ORDER)}

for df in [train, test]:
    df['시술 당시 나이']   = df['시술 당시 나이'].map(AGE_MAP).fillna(-1).astype(int)
    df['난자 기증자 나이'] = df['난자 기증자 나이'].map(DONOR_AGE_MAP).fillna(0).astype(int)
    df['정자 기증자 나이'] = df['정자 기증자 나이'].map(DONOR_AGE_MAP).fillna(0).astype(int)

print('✅ 나이 컬럼 Ordinal 인코딩 완료')
print('  AGE_MAP :', AGE_MAP)

✅ 나이 컬럼 Ordinal 인코딩 완료
  AGE_MAP : {'만18-34세': 0, '만35-37세': 1, '만38-39세': 2, '만40-42세': 3, '만43-44세': 4, '만45-50세': 5, '만50세 이상': 6}


### 7-2. 나머지 범주형 컬럼 — Label 인코딩
> train+test 전체로 fit → 미등장 값 `'__MISSING__'` 으로 안전 처리  
> LabelEncoder는 레이블 순서를 학습하지 않으므로 Data Leakage 없음

In [9]:
CAT_LABEL_COLS = [
    '시술 시기 코드', '시술 유형',
    '배란 유도 유형', '배아 생성 주요 이유',
    '난자 출처', '정자 출처',
    '특정 시술 유형',
]
CAT_LABEL_COLS = [c for c in CAT_LABEL_COLS if c in train.columns]

label_encoders = {}

for col in CAT_LABEL_COLS:
    le = LabelEncoder()
    combined = (
        pd.concat([train[col], test[col]], axis=0)
        .fillna('__MISSING__')
        .astype(str)
    )
    le.fit(combined)

    train[col] = le.transform(train[col].fillna('__MISSING__').astype(str))
    test[col]  = le.transform(test[col].fillna('__MISSING__').astype(str))

    label_encoders[col] = le

print('✅ Label Encoding 완료')
for col, le in label_encoders.items():
    print(f'  {col:<20} → {len(le.classes_)}개 클래스')

✅ Label Encoding 완료
  시술 시기 코드             → 7개 클래스
  시술 유형                → 2개 클래스
  배란 유도 유형             → 4개 클래스
  배아 생성 주요 이유          → 14개 클래스
  난자 출처                → 3개 클래스
  정자 출처                → 4개 클래스
  특정 시술 유형             → 27개 클래스


## 8. 최종 검증

In [10]:
feature_cols = [c for c in train.columns if c not in [TARGET, 'ID']]

X_train = train[feature_cols]
y_train = train[TARGET]
X_test  = test[feature_cols]

print('=' * 55)
print(f'X_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}')
print('=' * 55)
print(f'피처 수          : {len(feature_cols)}')
print(f'Train 잔여 결측  : {X_train.isnull().sum().sum()}')
print(f'Test  잔여 결측  : {X_test.isnull().sum().sum()}')
print(f'타겟 분포 (0/1)  : {y_train.value_counts().to_dict()}')
print('=' * 55)

# 파생 피처 포함 여부 확인
DERIVED_COLS = ['수정률', '이식률', '배양기간', '시술_임신_비율', 'has_blastocyst']
for fc in DERIVED_COLS:
    ok = '✅' if fc in feature_cols else '❌'
    print(f'  {ok} {fc}')

X_train : (256351, 81)  |  y_train : (256351,)
X_test  : (90067, 81)
피처 수          : 81
Train 잔여 결측  : 0
Test  잔여 결측  : 0
타겟 분포 (0/1)  : {0: 190123, 1: 66228}
  ✅ 수정률
  ✅ 이식률
  ✅ 배양기간
  ✅ 시술_임신_비율
  ✅ has_blastocyst


## 9. pkl 저장

In [11]:
# 전처리 완료 데이터
with open(f'{OUTPUT_DIR}/train_preprocessed.pkl', 'wb') as f:
    pickle.dump({'X': X_train, 'y': y_train, 'feature_cols': feature_cols}, f)

with open(f'{OUTPUT_DIR}/test_preprocessed.pkl', 'wb') as f:
    pickle.dump({'X': X_test, 'feature_cols': feature_cols}, f)

# 인코더 및 통계량 (재현성 보장)
with open(f'{OUTPUT_DIR}/encoders.pkl', 'wb') as f:
    pickle.dump({
        'label_encoders': label_encoders,
        'age_map':        AGE_MAP,
        'donor_age_map':  DONOR_AGE_MAP,
        'group_medians':  group_medians,
        'global_medians': global_medians,
        'feature_cols':   feature_cols,
    }, f)

print('✅ pkl 저장 완료')
print(f'  · {OUTPUT_DIR}/train_preprocessed.pkl')
print(f'  · {OUTPUT_DIR}/test_preprocessed.pkl')
print(f'  · {OUTPUT_DIR}/encoders.pkl')

✅ pkl 저장 완료
  · ../data/train_preprocessed.pkl
  · ../data/test_preprocessed.pkl
  · ../data/encoders.pkl
